<a href="https://colab.research.google.com/github/Skulfre/Machine-Learning/blob/main/Paper_Replicating.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Continue with regular imports
import matplotlib.pyplot as plt
import torch
import torchvision

from torch import nn
from torchvision import transforms

# Try to get torchinfo, install it if it doesn't work
try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

# Try to import the going_modular directory, download it from GitHub if it doesn't work
try:
    from going_modular.going_modular import data_setup, engine
    from helper_functions import download_data, set_seeds, plot_loss_curves
except:
    # Get the going_modular scripts
    print("[INFO] Couldn't find going_modular or helper_functions scripts... downloading them from GitHub.")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/going_modular .
    !mv pytorch-deep-learning/helper_functions.py . # get the helper_functions.py script
    !rm -rf pytorch-deep-learning
    from going_modular.going_modular import data_setup, engine
    from helper_functions import download_data, set_seeds, plot_loss_curves

In [ ]:
# Download pizza, steak, sushi images from GitHub
image_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                           destination="pizza_steak_sushi")
image_path

In [ ]:
train_dir = image_path / "train"
test_dir = image_path / "test"

In [ ]:
train_dir, test_dir

In [ ]:
from going_modular.going_modular import data_setup
from torchvision import transforms

IMG_SIZE = 224 # Table 3 ViT paper

# Create transforms pipeline
manual_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

BATCH_SIZE = 32

train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                              test_dir=test_dir,
                              transform=manual_transform,
                              batch_size=BATCH_SIZE)

In [ ]:
train_dataloader, test_dataloader, class_names

In [ ]:
image, label = next(iter(train_dataloader))
img = image[0]
label = label[0]

image, label


In [ ]:
import matplotlib.pyplot as plt

img_permuted = img.permute(1, 2, 0)
plt.figure(figsize=(10, 7))
plt.title(f"Class: {class_names[label]}")
plt.imshow(img_permuted)
plt.axis(False)

In [ ]:
patch_size = 16
plt.figure(figsize=(patch_size, patch_size))
plt.imshow(img_permuted[:patch_size, :,:])

In [ ]:
from matplotlib import figure
fig, ax = plt.subplots(nrows=1,
                       ncols=int(224 / patch_size),
                       sharex=True,
                       sharey=True,
                       figsize=(patch_size,patch_size))

for i, patch in enumerate(range(0, 224, patch_size)):
  ax[i].imshow(img_permuted[:patch_size, patch:patch+patch_size, :])
  plt.axis(False)

In [ ]:
img_size = 224
patch_size = 16
num_patches = img_size / patch_size
print(f"Number of patches per row: {num_patches}")
print(f"Number of patches per column: {num_patches}")
print(f"Total number of patches: {num_patches * num_patches}")
print(f"Patch size: {patch_size} pixels x {patch_size} pixels ")

fig, axs = plt.subplots(nrows=int(img_size/patch_size),
                        ncols=int(img_size/patch_size),
                        figsize=(num_patches, num_patches),
                        sharex=True,
                        sharey=True)

for i, patch_height in enumerate(range(0, img_size, patch_size)):
  for j, patch_width in enumerate(range(0, img_size, patch_size)):
    # Plot the permuted image on the different axes
    axs[i, j].imshow(img_permuted[patch_height:patch_height+patch_size,
                                  patch_width:patch_width+patch_size, :])

    # Set up label information for each suplot (patch)
    axs[i, j].set_ylabel(i+1,
                         rotation="horizontal",
                         horizontalalignment="right",
                         verticalalignment="center")
    axs[i, j].set_xlabel(j+1)
    axs[i, j].set_xticks([])
    axs[i, j].set_yticks([])
    axs[i, j].label_outer()

    # Set up a title for the plot
    fig.suptitle(f"{class_names[label]}")
    plt.show()

In [ ]:
# Create conv2d layer to turn image into patches of learnable feature maps (embeddings)

from torch import nn

pathc_size = 16

conv2d = nn.Conv2d(in_channels=3,
                   out_channels=768, # D size from table 1 (embedding size)
                   kernel_size=patch_size,
                   stride=patch_size,
                   padding=0)
conv2d

In [ ]:
# View single image
image_out_of_conv = conv2d(img.unsqueeze(dim=0))
image_out_of_conv.shape
# [batch_size, embedding_dim, feature_map_height, feature_map_width]

In [ ]:
flat_img = nn.Flatten(start_dim=2, end_dim=3)
image_out_of_flatten_layer = flat_img(image_out_of_conv)
image_out_of_flatten_layer_permuted = image_out_of_flatten_layer.permute(0, 2, 1)
print(f"Before permutation: {image_out_of_flatten_layer.shape}")
print(f"After permutationL {image_out_of_flatten_layer_permuted.shape}")
# Batch size, num_of_patches, embedding_dimension

In [ ]:
##  Turning the ViT patch embedding layer into a PyTorch module
import torch
from torch import nn

PATCH_SIZE = 16
IN_CHANNELS = 3
OUT_CHANNELS = 768

class PatchEmbedding(nn.Module):

  def __init__(self,
               in_channels:int=3,
               embedding_dim:int=768,
               patch_size:int=16):
    super().__init__()

    self.conv2d_layer = nn.Conv2d(in_channels=in_channels,
                                  out_channels=embedding_dim,
                                  kernel_size=patch_size,
                                  stride=patch_size,
                                  padding=0)

    self.flatten_layer = nn.Flatten(start_dim=2,
                                    end_dim=3)

  def forward(self, x: torch.Tensor) -> torch.Tensor:

    image_resolution = x.shape[-1]
    print(f"Input image shape: {x.shape}")
    assert int(image_resolution % patch_size) == 0, f"Input image size must be divisible by patch size, image shape: {image_resolution}, patch size: {patch_size}"

    image_thorugh_conv = self.conv2d_layer(x)
    print(f"Image through conv2d layer shape: {image_thorugh_conv.shape}")
    image_through_flatten = self.flatten_layer(image_thorugh_conv)
    print(f"Image through flatten layer shape: {image_through_flatten.shape}")
    print(f"Image permuted: {image_through_flatten.permute(0, 2, 1).shape} ")
    return image_through_flatten.permute(0, 2, 1)



In [ ]:
img.shape

In [ ]:
set_seeds()

# Create an instance of patch embedding layer
patchify = PatchEmbedding(in_channels=3,
                          embedding_dim=768,
                          patch_size=16)

# Pass a single image through it
# print(f"Input image size: {img.unsqueeze(dim=0).shape}")
patch_embedded_image = patchify(img.unsqueeze(dim=0))

print(f"Output patch embedding sequence shape: {patch_embedded_image.shape}")

In [ ]:
patch_embedded_image

In [ ]:
# Get the batch size and emdedding dimension

batch_size = patch_embedded_image.shape[0]
embedding_dimesion = patch_embedded_image.shape[-1]

batch_size, embedding_dimesion

In [ ]:
# Create class token embedding as a learnable parameter that shared the same size as the embedding dimension (D)

class_token = nn.Parameter(torch.ones(batch_size, 1, embedding_dimesion),
                           requires_grad=True)
# use .ones for better visualisation
class_token.shape

In [ ]:
patch_embedded_image.shape

In [ ]:
# Add the class token embedding to the front of the patch embedding
patch_embedded_image_with_class_embedding = torch.cat((class_token, patch_embedded_image),
                                                       dim=1)

print(patch_embedded_image_with_class_embedding)
print(patch_embedded_image_with_class_embedding.shape)
# Batch size, class_token + patch_size, embedding_dim

In [ ]:
# Creating the position embedding
patch_embedded_image_with_class_embedding.shape

In [ ]:
# Calculate N (number_of_patches)
height = 224
width = 224
number_of_patches = int((height*width) / patch_size**2)
number_of_patches

embedding_dimension = patch_embedded_image_with_class_embedding.shape[-1]
embedding_dimension

# Create the learnable 1D position embedding
position_embedding = nn.Parameter(torch.ones(1,
                                            number_of_patches+1,
                                            embedding_dimension))
position_embedding, position_embedding.shape

In [ ]:
# Add the position embedding to the patch and class token embedding
patch_and_position_embedding = patch_embedded_image_with_class_embedding + position_embedding
patch_and_position_embedding, patch_and_position_embedding.shape

In [ ]:
img.shape

In [ ]:
image, label = next(iter(train_dataloader))
img = image[0]
img.shape

In [ ]:
# Equation 1 - All together
set_seeds()
embedding_dim = 768
batch_size = 1
number_of_patches = int((height*width) / patch_size**2)
patchify = PatchEmbedding(in_channels=3,
                          embedding_dim=768,
                          patch_size=16)

patchified_image = patchify(img.unsqueeze(dim=0))

class_token = nn.Parameter(torch.randn(batch_size, 1, embedding_dim),
                           requires_grad=True)

patchified_image_ct = torch.cat((class_token, patchified_image), dim=1)
print(f"Class Token Patchified Shape: {patchified_image_ct.shape}")

embedding_pos = nn.Parameter(torch.randn(batch_size, number_of_patches + 1, embedding_dim), requires_grad=True)
patchified = patchified_image_ct + embedding_pos

print(f"Final Patchified Shape: {patchified.shape}")

In [ ]:
patchified

In [ ]:
"""
Normalization kod D (embedding dimension) jeste osigurati da svaki sample ima isti mean i std
Primjer: pokusavamo da sidjemo niz stepenice gde svaki stepenik ima drugaciju sirinu, visinu, krivudavost.
Bilo bi tesko zar ne? Zbunjujuce.
Sa normalization - on omugucava da svaki stepenik bude iste sirine, duzine. Cime dolazi da mozemo brzo,
lako i precizno da se spustimo niz stepenice.
"""

In [ ]:
batch_size = 1
seq_len = 197
embedding_dim = 768

d_model = 768
heads = d_model // 8

class MultiHeadSelfAttentionBlock(nn.Module):

  def __init__(self,
               num_heads:int = 12, # Heads from table 1 ViT Base,
               embedding_dim:int = 768, # -> Embedding dimension
               attn_dropout:int = 0 # -> For the base ViT there are no dropouts
               ):

    super().__init__()

    self.layernorm = nn.LayerNorm(normalized_shape=embedding_dim)

    self.mha = nn.MultiheadAttention(embed_dim=embedding_dim,
                                     num_heads=num_heads,
                                     dropout=attn_dropout,
                                     batch_first=True)


  def forward(self, x):
    x_norm = self.layernorm(x)
    attn_output, attn_output_weights = self.mha(query=x_norm,
                                                key=x_norm,
                                                value=x_norm,
                                                need_weights=False)
    # In this example we dont need attn_output_weights but yea
    return attn_output + x # Skip connection -> Adding and input to the output

mha_block = MultiHeadSelfAttentionBlock(num_heads = 12)
attentionfied = mha_block(patchified)



In [ ]:
attentionfied, attentionfied.shape

In [ ]:
from torch.nn.modules.linear import Linear
from torch.nn.modules.dropout import Dropout

class MultiLayerPerceptronBlock(nn.Module):

  def __init__(self,
               embedding_dim:int=768,
               hidden_layers:int=3072, # Table 1
               dropout:float=0.1 # Table 3
               ):

    super().__init__()

    self.layer_norm = nn.LayerNorm(normalized_shape=embedding_dim)

    self.mlp = nn.Sequential(
        nn.Linear(in_features=embedding_dim, out_features=hidden_layers),
        nn.GELU(),
        nn.Dropout(p=dropout),
        nn.Linear(in_features=hidden_layers, out_features=embedding_dim),
        nn.Dropout(p=dropout),
    )

  def forward(self, x):
    x_norm = self.layer_norm(x)
    x_mlp = self.mlp(x_norm)
    return x_mlp + x

mlp_block = MultiLayerPerceptronBlock()
multified = mlp_block(attentionfied)

In [ ]:
multified, multified.shape

In [ ]:
class TransformerEncoder(nn.Module):

    def __init__(
        self,
        num_heads:int = 12, # Heads from table 1 ViT Base,
        embedding_dim:int = 768, # -> Embedding dimension
        attn_dropout:float = 0, # -> For the base ViT there are no dropouts
        dropout:float = 0.1,
        hidden_layers:int = 3072):

      super().__init__()

      # MultiSelfAttetionBlock ------------------------------------------ START

      self.layer_norm1 = nn.LayerNorm(normalized_shape=embedding_dim)

      self.mha = nn.MultiheadAttention(embed_dim=embedding_dim,
                                        num_heads=num_heads,
                                        dropout=attn_dropout,
                                        batch_first=True)
      # MultiSelfAttetionBlock ------------------------------------------ END

      # MultiLayerPerceptronBlock ------------------------------------------ START

      self.layer_norm2 = nn.LayerNorm(normalized_shape=embedding_dim)

      self.mlp = nn.Sequential(
          nn.Linear(in_features=embedding_dim, out_features=hidden_layers),
          nn.GELU(),
          nn.Dropout(p=dropout),
          nn.Linear(in_features=hidden_layers, out_features=embedding_dim),
          nn.Dropout(p=dropout),
      )

      # MultiLayerPerceptronBlock ------------------------------------------ END

    def forward(self, x):

      # MultiSelfAttetionBlock ------------------------------------------ START

      x_norm = self.layer_norm1(x)
      attn_output, attn_output_weights = self.mha(query=x_norm,
                                                  key=x_norm,
                                                  value=x_norm,
                                                  need_weights=False)
      mha_output = attn_output + x

      # MultiSelfAttetionBlock ------------------------------------------ END

      # MultiLayerPerceptronBlock ------------------------------------------ START

      x_norm = self.layer_norm2(mha_output)
      x_mlp = self.mlp(x_norm)

      mlp_output = x_mlp + mha_output

      # MultiLayerPerceptronBlock ------------------------------------------ END

      return mlp_output

In [ ]:
Transformer_Encoder = TransformerEncoder()
x = Transformer_Encoder(patchified)

In [ ]:
x, x.shape

In [ ]:
summary(model=Transformer_Encoder)

In [ ]:
# Now that is a custom Transformer Encoder
# Lets now create a built in PyTorch Transformer Layer

In [ ]:
torch_transformer_encodes_layer = nn.TransformerEncoderLayer(d_model=768,
                                                              nhead=12,
                                                              dim_feedforward=3072,
                                                              dropout=0.1,
                                                              activation='gelu',
                                                              batch_first=True,
                                                              norm_first=True)
torch_transformer_encodes_layer

In [ ]:
summary(model=torch_transformer_encodes_layer)

In [ ]:
class ViT(nn.Module):
  def __init__(self,
               img_size:int=224, # Table 3 from ViT paper
               in_channels:int=3,
               patch_size:int=16,
               num_transformer_layers:int=12, # Table 1 for "Layers" for ViT-Base
               embedding_dim:int=768, # Hidden size D from Table 1 for ViT - Base
               hidden_units:int=3072,
               num_heads:int=12,
               attn_dropout:int=0,
               mlp_dropout:float=0.1,
               embedding_dropout:int=0.1, # Dropout for patch and position embeddings
               num_classes:int=1000):
    super().__init__()

    assert int(img_size % patch_size) == 0, f"Input image size must be divisible by patch size, image shape: {img_size}, patch size: {patch_size}"

    # Calculate the number of patches (height * width / patch**2)
    self.num_patches = (img_size * img_size) // patch_size**2

    # Create learnable class embedding
    self.class_embedding = nn.Parameter(data=torch.randn(1, 1, embedding_dim),
                                        requires_grad=True)

    # Create learnable possition embedding
    self.position_embedding = nn.Parameter(data=torch.randn(1, self.num_patches + 1, embedding_dim),
                                           requires_grad=True)

    # Create embedding dropout value
    self.embedding_dropout = nn.Dropout(p=embedding_dropout)

    # Create patch embedding layer
    self.patch_embedding = PatchEmbedding(in_channels=in_channels,
                                          embedding_dim=embedding_dim,
                                          patch_size=patch_size)

    # Create the transformer encoder block
    self.transformer_encoder = nn.Sequential(*[TransformerEncoder(embedding_dim=embedding_dim,
                                                                 num_heads=num_heads,
                                                                 attn_dropout=attn_dropout,
                                                                 dropout=mlp_dropout,
                                                                 hidden_layers=hidden_units) for _ in range(num_transformer_layers)])
    # Create classifier head
    self.classifier = nn.Sequential(
        nn.LayerNorm(normalized_shape=embedding_dim),
        nn.Linear(in_features=embedding_dim,
                  out_features=num_classes)
    )

  def forward(self, x):
    batch_size = x.shape[0]
    class_token = self.class_embedding.expand(batch_size, -1, -1) # Adding a batch size on the 0th dimension (We are doing 32 images on 1 batch)
    x = self.patch_embedding(x)
    x_class_embedding = torch.cat((class_token, x), dim=1)
    x_position_emebding = x_class_embedding + self.position_embedding
    x_dropout = self.embedding_dropout(x_position_emebding)
    x = self.transformer_encoder(x_dropout)

    # Put the 0th index logit through classifier (equation 4)
    x = self.classifier(x[:, 0])
    return x

In [ ]:
vit_base = ViT()

In [ ]:
vit_base

In [ ]:
summary(model=vit_base)

In [ ]:
img.shape

In [ ]:
set_seeds()

random_image_tensor = torch.randn(1, 3, 224, 224)

vit = ViT(num_classes=len(class_names))

vit(img.unsqueeze(0))





In [ ]:
summary(model=ViT(),
        input_size=(1, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

In [ ]:
optimizer = torch.optim.Adam(params=vit.parameters(),
                             lr=1e-3,
                             weight_decay=0.1)

loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
from going_modular.going_modular import engine

set_seeds()
results = engine.train(model=vit,
                       train_dataloader=train_dataloader,
                       test_dataloader=test_dataloader,
                       epochs=10,
                       optimizer=optimizer,
                       loss_fn=loss_fn,
                       device=device)

In [ ]:
# Creating a built in ViT PyTorch

weights_pretrained = torchvision.models.ViT_B_16_Weights.DEFAULT
vit_pretrained = torchvision.models.vit_b_16(weights=weights_pretrained)

vit_pretrained

In [ ]:
vit_pretrained.heads

In [ ]:
summary(vit_pretrained)

In [ ]:
for params in vit_pretrained.parameters():
  params.requires_grad = False

summary(vit_pretrained)

In [ ]:
vit_pretrained.heads = nn.Sequential(
    nn.Linear(in_features=768, out_features=3)
)

summary(vit_pretrained)

In [ ]:
auto_transform = weights_pretrained.transforms()
auto_transform

In [ ]:
train_dir, test_dir

In [ ]:
from going_modular.going_modular import data_setup

train_dataloader_pretrained, test_dataloader_pretrained, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                                                                                                     test_dir=test_dir,
                                                                                                     transform=auto_transform,
                                                                                                     batch_size=32)

train_dataloader_pretrained, test_dataloader_pretrained, class_names

In [ ]:
optimizer = torch.optim.Adam(params=vit_pretrained.parameters(), lr=1e-3)

loss_fn = nn.CrossEntropyLoss()

In [ ]:
from going_modular.going_modular import engine

vit_pretrained_results = engine.train(model=vit_pretrained,
                                      train_dataloader=train_dataloader_pretrained,
                                      test_dataloader=test_dataloader_pretrained,
                                      optimizer=optimizer,
                                      loss_fn=loss_fn,
                                      epochs=10,
                                      device=device)

In [ ]:
from helper_functions import plot_loss_curves
plot_loss_curves(vit_pretrained_results)